In [21]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Domain Rental Analysis")
    .getOrCreate()
)

# Read rental data
rentals = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("domain/Data/vic_rentals_all.csv")
)

# Check data
rentals.printSchema()
rentals.show(5, truncate=False)

print("Number of rows:", rentals.count())
print("Number of columns:", len(rentals.columns))

root
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: string (nullable = true)
 |-- bond: double (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- days_listed: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- scraped_date: timestamp (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: double (nullable = true)
 |-- video_count: double (nullable = true)
 |-- floorplans_count: double (nullable = true)
 |-- virtual_tour: boolean (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (nu

In [22]:
# only keep data from 2025
from pyspark.sql import functions as F

rentals_2025 = rentals.filter(
    F.year("date_listed") == 2025
)

print("Original rows:", rentals.count())
print("2025 rows:", rentals_2025.count())
# count how many rows removed (not in 2025)
print("Removed:", rentals.count()-rentals_2025.count())

Original rows: 12738
2025 rows: 12064
Removed: 674


In [23]:
# convert data type for `available_date`
rentals_2025 = rentals_2025.withColumn(
    "available_date_clean",
    F.regexp_replace(
        F.col("available_date"),
        r"^[A-Za-z]+,\s*",
        ""
    )
)

rentals_2025 = rentals_2025.withColumn(
    "available_date_clean",
    F.to_date(
        F.col("available_date_clean"),
        "dd MMMM yyyy"
    )
)

# Create weekday column from cleaned available date
rentals_2025 = rentals_2025.withColumn(
    "available_weekday",
    F.date_format(
        F.col("available_date_clean"),
        "EEEE"
    )
)

rentals_2025.select(
    "available_date",
    "available_date_clean",
    "available_weekday",
    "date_listed"
).show(10, truncate=False)

+--------------------------+--------------------+-----------------+-----------+
|available_date            |available_date_clean|available_weekday|date_listed|
+--------------------------+--------------------+-----------------+-----------+
|Tuesday, 02 September 2025|2025-09-02          |Tuesday          |2025-08-13 |
|Thursday, 27 March 2025   |2025-03-27          |Thursday         |2025-03-06 |
|Monday, 15 September 2025 |2025-09-15          |Monday           |2025-08-19 |
|Wednesday, 20 August 2025 |2025-08-20          |Wednesday        |2025-08-21 |
|Tuesday, 02 September 2025|2025-09-02          |Tuesday          |2025-09-03 |
|Tuesday, 16 September 2025|2025-09-16          |Tuesday          |2025-09-08 |
|Thursday, 29 May 2025     |2025-05-29          |Thursday         |2025-04-10 |
|Monday, 06 May 2024       |2024-05-06          |Monday           |2025-04-16 |
|Thursday, 17 July 2025    |2025-07-17          |Thursday         |2025-06-19 |
|Monday, 11 August 2025    |2025-08-11  

In [24]:
# Total number of observations
n = rentals_2025.count()

# Missing count for each variable
missing_counts = rentals_2025.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in rentals_2025.columns
]).collect()[0].asDict()

# Create a summary table
missing_summary = [
    (
        c,
        missing_counts[c],
        round(missing_counts[c] / n * 100, 2)
    )
    for c in rentals_2025.columns
]

missing_df = spark.createDataFrame(
    missing_summary,
    ["variable", "missing_count", "missing_percent"]
)

# Show variables with the most missing values first
missing_df.orderBy(
    F.desc("missing_percent")
).show(50, truncate=False)

+--------------------+-------------+---------------+
|variable            |missing_count|missing_percent|
+--------------------+-------------+---------------+
|land_area           |12062        |99.98          |
|property_id         |9906         |82.11          |
|carspaces           |1613         |13.37          |
|structured_features |1186         |9.83           |
|bond                |656          |5.44           |
|agent_names         |164          |1.36           |
|bedrooms            |76           |0.63           |
|address             |69           |0.57           |
|weekly_rent         |53           |0.44           |
|available_date      |26           |0.22           |
|available_date_clean|26           |0.22           |
|available_weekday   |26           |0.22           |
|bathrooms           |16           |0.13           |
|url                 |7            |0.06           |
|lat                 |3            |0.02           |
|lon                 |3            |0.02      

In [25]:
# Metadata / identifier columns
# Retained for identification, validation, geographic analysis,
# or reference, but not intended as direct model predictors.
metadata_cols = [
    "listing_id",
    "address",
    "url",
    "domain_page_id",
    "agency_id",
    "agent_names"
]

# Geographic information
geographic_cols = [
    "suburb",
    "postcode",
    "lat",
    "lon"
]

In [26]:
# remove `land_area` and `property_id`
rentals_2025 = rentals_2025.drop(
    "land_area",
    "property_id"
)


In [29]:
# check if there is any `weekly_rent` Not NULL but cannot be converted to double

rentals_2025.filter(
    F.col("weekly_rent").isNotNull() &
    F.col("weekly_rent").cast("double").isNull()
).select(
    "weekly_rent"
).distinct().show(50, truncate=False)

# convert from string to double
rentals_2025 = rentals_2025.withColumn(
    "weekly_rent",
    F.col("weekly_rent").cast("double")
)
rentals_2025.select("weekly_rent").printSchema()

+-----------+
|weekly_rent|
+-----------+
+-----------+

root
 |-- weekly_rent: double (nullable = true)



In [31]:
before = rentals_2025.count()

rentals_2025 = rentals_2025.filter(
    F.col("weekly_rent").isNotNull()
)

after = rentals_2025.count()

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

rentals_2025.select("weekly_rent").summary().show()

# Highest weekly rents
rentals_2025.select(
    "listing_id",
    "weekly_rent",
    "property_type",
    "bedrooms",
    "suburb",
    "address"
).orderBy(
    F.desc("weekly_rent")
).show(20, truncate=False)

# Lowest weekly rents
rentals_2025.select(
    "listing_id",
    "weekly_rent",
    "property_type",
    "bedrooms",
    "suburb",
    "address"
).orderBy(
    F.asc("weekly_rent")
).show(20, truncate=False)

Rows before: 12011
Rows after: 12011
Rows removed: 0
+-------+-----------------+
|summary|      weekly_rent|
+-------+-----------------+
|  count|            12011|
|   mean|769.3124635750562|
| stddev|9594.306956153023|
|    min|              0.0|
|    25%|            490.0|
|    50%|            560.0|
|    75%|            690.0|
|    max|         808500.0|
+-------+-----------------+

+----------+-----------+-----------------------+--------+---------------+-----------------------+
|listing_id|weekly_rent|property_type          |bedrooms|suburb         |address                |
+----------+-----------+-----------------------+--------+---------------+-----------------------+
|17697966  |808500.0   |House                  |4.0     |BERWICK        |83 Golf Links Road     |
|17750082  |595000.0   |House                  |3.0     |ROSEBUD        |4 Wilfred Street       |
|17747509  |315000.0   |Apartment / Unit / Flat|1.0     |MORWELL        |1/28 Elgin Street      |
|17712191  |10000.0   

In [33]:
# inspect these extreme data in detail
rentals_2025.filter(
    F.col("weekly_rent") > 100000
).select(
    "listing_id",
    "weekly_rent",
    "bond",
    "property_type",
    "bedrooms",
    "bathrooms",
    "suburb",
    "address",
    "url"
).show(truncate=False)


# three rows with abnormal extremly high `weekly_rent`
invalid_rent_ids = [
    "17747509",
    "17697966",
    "17750082"
]

# remove these invalid and abnormal rows
rentals_2025 = rentals_2025.filter(
    (F.col("weekly_rent") > 0) &
    (~F.col("listing_id").isin(invalid_rent_ids))
)

+----------+-----------+------+-----------------------+--------+---------+-------+------------------+----------------------------------------------------------------------+
|listing_id|weekly_rent|bond  |property_type          |bedrooms|bathrooms|suburb |address           |url                                                                   |
+----------+-----------+------+-----------------------+--------+---------+-------+------------------+----------------------------------------------------------------------+
|17747509  |315000.0   |1365.0|Apartment / Unit / Flat|1.0     |1.0      |MORWELL|1/28 Elgin Street |https://www.domain.com.au/1-28-elgin-street-morwell-vic-3840-17747509 |
|17697966  |808500.0   |2607.0|House                  |4.0     |2.0      |BERWICK|83 Golf Links Road|https://www.domain.com.au/83-golf-links-road-berwick-vic-3806-17697966|
|17750082  |595000.0   |2578.0|House                  |3.0     |2.0      |ROSEBUD|4 Wilfred Street  |https://www.domain.com.au/4-wilfre

Weekly rent was converted from string to numeric after confirming that all non-missing values were numeric. Listings with missing or non-positive weekly rent were removed. Three listings with implausibly high weekly rents ($315,000–$808,500 per week) were identified through manual inspection. Their reported bond amounts were inconsistent with these rental values, indicating likely data extraction errors, so these records were removed. Other high rental values were retained because they may represent genuine luxury properties.

In [34]:
rentals_2025.select("weekly_rent").summary().show()

+-------+------------------+
|summary|       weekly_rent|
+-------+------------------+
|  count|             12003|
|   mean| 626.6526701657918|
| stddev|288.95086805165045|
|    min|              27.0|
|    25%|             490.0|
|    50%|             560.0|
|    75%|             690.0|
|    max|           10000.0|
+-------+------------------+



In [35]:
# Check duplicated listing IDs
duplicate_ids = (
    rentals_2025
    .groupBy("listing_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicated listing IDs:", duplicate_ids.count())

# Check completely duplicated rows
print(
    "Completely duplicated rows:",
    rentals_2025.count() - rentals_2025.dropDuplicates().count()
)

Duplicated listing IDs: 0
Completely duplicated rows: 0


In [36]:
rentals_2025.groupBy(
    "property_type"
).count().orderBy(
    F.desc("count")
).show(50, truncate=False)

rentals_2025.groupBy(
    "primary_type"
).count().orderBy(
    F.desc("count")
).show(50, truncate=False)

+-----------------------------+-----+
|property_type                |count|
+-----------------------------+-----+
|House                        |6274 |
|Apartment / Unit / Flat      |4240 |
|Townhouse                    |1215 |
|Studio                       |169  |
|New Apartments / Off the Plan|40   |
|Villa                        |25   |
|Acreage / Semi-Rural         |10   |
|Semi-Detached                |9    |
|New House & Land             |7    |
|Car Space                    |3    |
|Block of Units               |3    |
|Terrace                      |3    |
|Farm                         |2    |
|Duplex                       |2    |
|Vacant land                  |1    |
+-----------------------------+-----+

+----------------+-----+
|primary_type    |count|
+----------------+-----+
|House           |6284 |
|Apartment       |4412 |
|Townhouse/Villa |1240 |
|New Developments|40   |
|Any             |26   |
|Land            |1    |
+----------------+-----+



In [37]:
# Keep only the four main primary property types
keep_types = [
    "House",
    "Apartment",
    "Townhouse/Villa",
    "New Developments"
]

before = rentals_2025.count()

rentals_2025 = rentals_2025.filter(
    F.col("primary_type").isin(keep_types)
)

after = rentals_2025.count()

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

# Check
rentals_2025.groupBy("primary_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

Rows before: 12003
Rows after: 11976
Rows removed: 27
+----------------+-----+
|    primary_type|count|
+----------------+-----+
|           House| 6284|
|       Apartment| 4412|
| Townhouse/Villa| 1240|
|New Developments|   40|
+----------------+-----+



In [39]:
total_rows = rentals_2025.count()

missing_summary = []

for c in rentals_2025.columns:
    missing_count = rentals_2025.filter(
        F.col(c).isNull()
    ).count()

    missing_summary.append(
        (c, missing_count, missing_count / total_rows * 100)
    )



missing_df = spark.createDataFrame(
    missing_summary,
    ["variable", "missing_count", "missing_percent"]
).withColumn(
    "missing_percent",
    F.round(F.col("missing_percent"), 4)
)

missing_df.orderBy(
    F.desc("missing_percent")
).show(50, truncate=False)

+--------------------+-------------+---------------+
|variable            |missing_count|missing_percent|
+--------------------+-------------+---------------+
|carspaces           |1591         |13.2849        |
|structured_features |1179         |9.8447         |
|bond                |652          |5.4442         |
|agent_names         |160          |1.336          |
|bedrooms            |68           |0.5678         |
|address             |67           |0.5595         |
|bathrooms           |10           |0.0835         |
|url                 |7            |0.0585         |
|lat                 |3            |0.0251         |
|lon                 |3            |0.0251         |
|agency              |1            |0.0084         |
|agency_id           |1            |0.0084         |
|listing_id          |0            |0.0            |
|suburb              |0            |0.0            |
|postcode            |0            |0.0            |
|weekly_rent         |0            |0.0       

In [41]:
check_cols = [
    "carspaces",
    "bond",
    "bedrooms",
    "bathrooms"
]

for c in check_cols:
    print(f"\nMissing summary for {c}:")

    rentals_2025.groupBy("primary_type").agg(
        F.count("*").alias("total"),
        F.sum(F.col(c).isNull().cast("int")).alias("missing_count")
    ).withColumn(
        "missing_percent",
        F.round(
            F.col("missing_count") / F.col("total") * 100,
            4
        )
    ).orderBy(
        F.desc("missing_percent")
    ).show()


Missing summary for carspaces:
+----------------+-----+-------------+---------------+
|    primary_type|total|missing_count|missing_percent|
+----------------+-----+-------------+---------------+
|New Developments|   40|           40|          100.0|
|       Apartment| 4412|         1052|        23.8441|
|           House| 6284|          477|         7.5907|
| Townhouse/Villa| 1240|           22|         1.7742|
+----------------+-----+-------------+---------------+


Missing summary for bond:
+----------------+-----+-------------+---------------+
|    primary_type|total|missing_count|missing_percent|
+----------------+-----+-------------+---------------+
|       Apartment| 4412|          345|         7.8196|
|           House| 6284|          260|         4.1375|
| Townhouse/Villa| 1240|           47|         3.7903|
|New Developments|   40|            0|            0.0|
+----------------+-----+-------------+---------------+


Missing summary for bedrooms:
+----------------+-----+----

Missing car-space information was retained rather than imputed, as missingness varied substantially by property type and a missing value could not be assumed to represent zero car spaces.

In [43]:
rentals_2025.select(
    F.corr("bond", "weekly_rent").alias("corr_bond_rent")
).show()

rentals_2025.select(
    "bond", "weekly_rent"
).filter(
    F.col("bond").isNotNull()
).summary().show()

+------------------+
|    corr_bond_rent|
+------------------+
|0.5778042246165429|
+------------------+

+-------+------------------+-----------------+
|summary|              bond|      weekly_rent|
+-------+------------------+-----------------+
|  count|             11324|            11324|
|   mean|2785.5204874602614|624.7539738608266|
| stddev| 2527.886516103497|290.4195816222883|
|    min|             150.0|             27.0|
|    25%|            2086.0|            490.0|
|    50%|            2433.0|            560.0|
|    75%|            2955.0|            680.0|
|    max|          212917.0|          10000.0|
+-------+------------------+-----------------+



In [44]:
rentals_2025.filter(
    F.col("bond").isNotNull()
).select(
    "listing_id",
    "weekly_rent",
    "bond",
    "primary_type",
    "property_type",
    "bedrooms",
    "suburb",
    "address"
).orderBy(
    F.desc("bond")
).show(20, truncate=False)

+----------+-----------+--------+---------------+-----------------------+--------+---------------+------------------------+
|listing_id|weekly_rent|bond    |primary_type   |property_type          |bedrooms|suburb         |address                 |
+----------+-----------+--------+---------------+-----------------------+--------+---------------+------------------------+
|17591648  |490.0      |212917.0|Apartment      |Apartment / Unit / Flat|2.0     |BELMONT        |4/367 High Street       |
|17712191  |10000.0    |60000.0 |House          |House                  |6.0     |TOORAK         |NULL                    |
|17722803  |5750.0     |34500.0 |House          |House                  |4.0     |TOORAK         |2 Lisbuoy Court         |
|17694334  |4500.0     |27000.0 |Apartment      |Apartment / Unit / Flat|3.0     |SOUTH YARRA    |1 Fairlie Court         |
|17744678  |4500.0     |27000.0 |House          |House                  |5.0     |HAWTHORN       |94 Illawarra Road       |
|1775096

One implausible bond value ($212,917 for a weekly rent of $490) was identified and replaced with a missing value rather than removing the entire listing.


In [45]:
# ============================================================
# CLEAN INVALID BOND VALUE
# ============================================================

# The bond value for listing 17591648 is implausible
# (bond = 212917 while weekly_rent = 490).
# Keep the listing, but replace the invalid bond with NULL
# so it can be handled together with other missing bond values.

rentals_2025 = rentals_2025.withColumn(
    "bond",
    F.when(
        F.col("listing_id") == "17591648",
        F.lit(None).cast("double")
    ).otherwise(F.col("bond"))
)

# Check the corrected record
rentals_2025.filter(
    F.col("listing_id") == "17591648"
).select(
    "listing_id",
    "weekly_rent",
    "bond",
    "property_type",
    "suburb"
).show(truncate=False)

# Check updated number of missing bond values
print(
    "Missing bond values:",
    rentals_2025.filter(F.col("bond").isNull()).count()
)

# Re-check bond distribution
rentals_2025.select("bond").summary().show()

# Re-check correlation with weekly rent
rentals_2025.select(
    F.round(
        F.corr("bond", "weekly_rent"),
        4
    ).alias("corr_bond_rent")
).show()

+----------+-----------+----+-----------------------+-------+
|listing_id|weekly_rent|bond|property_type          |suburb |
+----------+-----------+----+-----------------------+-------+
|17591648  |490.0      |NULL|Apartment / Unit / Flat|BELMONT|
+----------+-----------+----+-----------------------+-------+

Missing bond values: 653
+-------+------------------+
|summary|              bond|
+-------+------------------+
|  count|             11323|
|   mean| 2766.962554093438|
| stddev|1578.1254053781865|
|    min|             150.0|
|    25%|            2086.0|
|    50%|            2433.0|
|    75%|            2955.0|
|    max|           60000.0|
+-------+------------------+

+--------------+
|corr_bond_rent|
+--------------+
|        0.9311|
+--------------+



after remove that extreme row, the correlation increased from 0.5778 to 0.9311, and stddev decreased from 2527.8865 to 1578.1254.

In [47]:
# ============================================================
# COMPARE MODELS FOR BOND IMPUTATION
# WITHOUT WEEKLY_RENT TO AVOID TARGET LEAKAGE
# ============================================================

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor,
    GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator


# ------------------------------------------------------------
# 1. Use rows with observed bond values
# ------------------------------------------------------------

bond_known = rentals_2025.filter(
    F.col("bond").isNotNull()
)

bond_train, bond_test = bond_known.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", bond_train.count())
print("Testing rows:", bond_test.count())


# ------------------------------------------------------------
# 2. Encode primary property type
# ------------------------------------------------------------

primary_type_indexer = StringIndexer(
    inputCol="primary_type",
    outputCol="primary_type_index",
    handleInvalid="keep"
)

primary_type_encoder = OneHotEncoder(
    inputCol="primary_type_index",
    outputCol="primary_type_encoded"
)


# ------------------------------------------------------------
# 3. Assemble predictors
#
# weekly_rent is intentionally EXCLUDED because weekly_rent
# will be the target variable in downstream modelling.
# ------------------------------------------------------------

assembler = VectorAssembler(
    inputCols=[
        "days_listed",
        "photo_count",
        "video_count",
        "floorplans_count",
        "primary_type_encoded"
    ],
    outputCol="features"
)


# ------------------------------------------------------------
# 4. Candidate regression models
# ------------------------------------------------------------

models = {
    "Linear Regression": LinearRegression(
        featuresCol="features",
        labelCol="bond",
        predictionCol="prediction"
    ),

    "Random Forest": RandomForestRegressor(
        featuresCol="features",
        labelCol="bond",
        predictionCol="prediction",
        numTrees=100,
        maxDepth=8,
        seed=42
    ),

    "GBT": GBTRegressor(
        featuresCol="features",
        labelCol="bond",
        predictionCol="prediction",
        maxIter=100,
        maxDepth=5,
        stepSize=0.05,
        seed=42
    )
}


# ------------------------------------------------------------
# 5. Evaluators
# ------------------------------------------------------------

rmse_evaluator = RegressionEvaluator(
    labelCol="bond",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="bond",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="bond",
    predictionCol="prediction",
    metricName="r2"
)


# ------------------------------------------------------------
# 6. Fit and evaluate each model
# ------------------------------------------------------------

results = []

for model_name, regressor in models.items():

    pipeline = Pipeline(
        stages=[
            primary_type_indexer,
            primary_type_encoder,
            assembler,
            regressor
        ]
    )

    fitted_model = pipeline.fit(bond_train)

    predictions = fitted_model.transform(bond_test)

    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    results.append(
        (model_name, rmse, mae, r2)
    )


# ------------------------------------------------------------
# 7. Compare model performance
# ------------------------------------------------------------

results_df = spark.createDataFrame(
    results,
    ["model", "RMSE", "MAE", "R2"]
)

results_df = (
    results_df
    .withColumn("RMSE", F.round("RMSE", 4))
    .withColumn("MAE", F.round("MAE", 4))
    .withColumn("R2", F.round("R2", 4))
    .orderBy("MAE")
)

results_df.show(truncate=False)


# ------------------------------------------------------------
# 8. Select best model based primarily on MAE
# ------------------------------------------------------------

best_model = results_df.first()

print(f"Best model based on MAE: {best_model['model']}")

print(
    f"MAE = {best_model['MAE']}, "
    f"RMSE = {best_model['RMSE']}, "
    f"R² = {best_model['R2']}"
)

Training rows: 9136


Testing rows: 2187


26/09/09 17:48:51 WARN Instrumentation: [d39d9dba] regParam is zero, which might cause numerical instability and overfitting.
26/09/09 17:48:52 WARN Instrumentation: [d39d9dba] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/09/09 17:48:59 WARN DAGScheduler: Broadcasting large task binary with size 1495.3 KiB
26/09/09 17:49:00 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB


+-----------------+---------+--------+------+
|model            |RMSE     |MAE     |R2    |
+-----------------+---------+--------+------+
|Random Forest    |1882.7607|788.8759|0.0748|
|Linear Regression|1897.1223|802.5991|0.0607|
|GBT              |1929.877 |817.1379|0.028 |
+-----------------+---------+--------+------+

Best model based on MAE: Random Forest
MAE = 788.8759, RMSE = 1882.7607, R² = 0.0748


Model-based imputation was investigated for the missing bond values. Linear Regression, Random Forest, and Gradient-Boosted Trees (GBT) were compared using an 80/20 train-test split. `weekly_rent` was excluded from the predictors because it is the downstream prediction target, and using it to impute bond would introduce target leakage.

All three models showed poor predictive performance. Random Forest performed best, but achieved only R² = 0.075 (MAE = $788.88), indicating that the available predictors could not reliably estimate bond values.

Therefore, model-based imputation was not used. The 653 missing bond values were instead imputed using the median bond within each `primary_type`. An indicator variable, `bond_imputed`, was retained to identify imputed observations.

In [49]:
# Bond imputation using median within each primary property type

# Flag missing bond values
rentals_2025 = rentals_2025.withColumn(
    "bond_imputed",
    F.col("bond").isNull().cast("int")
)

# Calculate median bond by primary type
bond_medians = (
    rentals_2025
    .filter(F.col("bond").isNotNull())
    .groupBy("primary_type")
    .agg(
        F.percentile_approx("bond", 0.5).alias("median_bond")
    )
)

bond_medians.show()

# Impute missing bond values
rentals_2025 = (
    rentals_2025
    .join(bond_medians, on="primary_type", how="left")
    .withColumn(
        "bond",
        F.coalesce(F.col("bond"), F.col("median_bond"))
    )
    .drop("median_bond")
)

# Check result
rentals_2025.select(
    F.sum(F.col("bond").isNull().cast("int")).alias("missing_bond"),
    F.sum("bond_imputed").alias("imputed_bond")
).show()

rentals_2025.select("bond").summary().show()

+----------------+-----------+
|    primary_type|median_bond|
+----------------+-----------+
|       Apartment|     2390.0|
| Townhouse/Villa|     2824.0|
|New Developments|     1955.0|
|           House|     2390.0|
+----------------+-----------+

+------------+------------+
|missing_bond|imputed_bond|
+------------+------------+
|           0|           0|
+------------+------------+

+-------+------------------+
|summary|              bond|
+-------+------------------+
|  count|             11976|
|   mean|  2748.11163994656|
| stddev|1536.7243435426292|
|    min|             150.0|
|    25%|            2129.0|
|    50%|            2390.0|
|    75%|            2911.0|
|    max|           60000.0|
+-------+------------------+



In [51]:
# Missing bedrooms percentage by property type

rentals_2025.groupBy("property_type").agg(
    F.count("*").alias("total"),
    F.sum(F.col("bedrooms").isNull().cast("int")).alias("missing_count")
).withColumn(
    "missing_percent",
    F.round(F.col("missing_count") / F.col("total") * 100, 4)
).filter(
    F.col("missing_count") > 0
).orderBy(
    F.desc("missing_percent")
).show(truncate=False)

# Missing bathrooms percentage by property type

rentals_2025.groupBy("property_type").agg(
    F.count("*").alias("total"),
    F.sum(F.col("bathrooms").isNull().cast("int")).alias("missing_count")
).withColumn(
    "missing_percent",
    F.round(F.col("missing_count") / F.col("total") * 100, 4)
).filter(
    F.col("missing_count") > 0
).orderBy(
    F.desc("missing_percent")
).show(truncate=False)

+-----------------------+-----+-------------+---------------+
|property_type          |total|missing_count|missing_percent|
+-----------------------+-----+-------------+---------------+
|Studio                 |169  |43           |25.4438        |
|Apartment / Unit / Flat|4240 |19           |0.4481         |
|House                  |6274 |6            |0.0956         |
+-----------------------+-----+-------------+---------------+

+-----------------------+-----+-------------+---------------+
|property_type          |total|missing_count|missing_percent|
+-----------------------+-----+-------------+---------------+
|House                  |6274 |6            |0.0956         |
|Apartment / Unit / Flat|4240 |4            |0.0943         |
+-----------------------+-----+-------------+---------------+



In [52]:
# Set missing bedrooms in Studio properties to 0

rentals_2025 = rentals_2025.withColumn(
    "bedrooms",
    F.when(
        (F.col("property_type") == "Studio") &
        F.col("bedrooms").isNull(),
        0.0
    ).otherwise(F.col("bedrooms"))
)



In [53]:
# Recheck missing bedrooms and bathrooms

rentals_2025.select(
    F.sum(F.col("bedrooms").isNull().cast("int")).alias("missing_bedrooms"),
    F.sum(F.col("bathrooms").isNull().cast("int")).alias("missing_bathrooms")
).show()

+----------------+-----------------+
|missing_bedrooms|missing_bathrooms|
+----------------+-----------------+
|              25|               10|
+----------------+-----------------+



In [55]:
# Check overlap between missing bedrooms and bathrooms

rentals_2025.select(
    F.sum(
        (F.col("bedrooms").isNull() & F.col("bathrooms").isNull()).cast("int")
    ).alias("both_missing"),
    
    F.sum(
        (F.col("bedrooms").isNull() | F.col("bathrooms").isNull()).cast("int")
    ).alias("rows_with_missing")
).show()

# Remove remaining rows with missing bedrooms or bathrooms

before = rentals_2025.count()

rentals_2025 = rentals_2025.filter(
    F.col("bedrooms").isNotNull() &
    F.col("bathrooms").isNotNull()
)

after = rentals_2025.count()

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)
print("Percentage removed:", round((before - after) / before * 100, 4), "%")

+------------+-----------------+
|both_missing|rows_with_missing|
+------------+-----------------+
|           7|               28|
+------------+-----------------+

Rows before: 11976
Rows after: 11948
Rows removed: 28
Percentage removed: 0.2338 %


Missing bedrooms for Studio properties were set to 0, reflecting the absence of a separate bedroom. After this adjustment, only 28 observations (0.23%) had missing bedrooms and/or bathrooms. Given the very small proportion, these observations were removed rather than imputed.

In [57]:
# Carspaces distribution by primary property type

rentals_2025.groupBy("primary_type").agg(
    F.count("*").alias("total"),
    F.sum(F.col("carspaces").isNull().cast("int")).alias("missing"),
    F.round(
        F.sum(F.col("carspaces").isNull().cast("int")) /
        F.count("*") * 100, 4
    ).alias("missing_percent"),
    F.percentile_approx("carspaces", 0.5).alias("median_carspaces"),
    F.avg("carspaces").alias("mean_carspaces")
).orderBy("primary_type").show(truncate=False)

# Carspaces value distribution

rentals_2025.filter(
    F.col("carspaces").isNotNull()
).groupBy(
    "primary_type", "carspaces"
).count().orderBy(
    "primary_type", "carspaces"
).show(100, truncate=False)

+----------------+-----+-------+---------------+----------------+------------------+
|primary_type    |total|missing|missing_percent|median_carspaces|mean_carspaces    |
+----------------+-----+-------+---------------+----------------+------------------+
|Apartment       |4391 |1034   |23.5482        |1.0             |1.140601727733095 |
|House           |6277 |475    |7.5673         |2.0             |1.9820751465012065|
|New Developments|40   |40     |100.0          |NULL            |NULL              |
|Townhouse/Villa |1240 |22     |1.7742         |2.0             |1.5788177339901477|
+----------------+-----+-------+---------------+----------------+------------------+

+---------------+---------+-----+
|primary_type   |carspaces|count|
+---------------+---------+-----+
|Apartment      |1.0      |2921 |
|Apartment      |2.0      |412  |
|Apartment      |3.0      |18   |
|Apartment      |4.0      |4    |
|Apartment      |6.0      |1    |
|Apartment      |8.0      |1    |
|House       

In [59]:
# Impute missing carspaces by property type

rentals_2025 = rentals_2025.withColumn(
    "carspaces_imputed",
    F.col("carspaces").isNull().cast("int")
)

rentals_2025 = rentals_2025.withColumn(
    "carspaces",
    F.when(
        F.col("carspaces").isNull() &
        (F.col("primary_type") == "Apartment"),
        1.0
    ).when(
        F.col("carspaces").isNull() &
        F.col("primary_type").isin("House", "Townhouse/Villa"),
        2.0
    ).when(
        F.col("carspaces").isNull() &
        (F.col("primary_type") == "New Developments"),
        0.0
    ).otherwise(F.col("carspaces"))
)

# Check result
rentals_2025.select(
    F.sum(F.col("carspaces").isNull().cast("int")).alias("missing_carspaces"),
    F.sum("carspaces_imputed").alias("imputed_carspaces")
).show()

+-----------------+-----------------+
|missing_carspaces|imputed_carspaces|
+-----------------+-----------------+
|                0|             1571|
+-----------------+-----------------+



### Carspaces Imputation

Missing `carspaces` values showed different patterns across property types. Apartments typically had 1 carspace, while houses and townhouses/villas typically had 2. Therefore, missing values were imputed using these group-specific typical values.

All New Developments had missing `carspaces`, so no group-specific value could be estimated. As these listings provided no recorded carspace information, their missing values were conservatively set to 0.

A `carspaces_imputed` indicator was retained to distinguish imputed values from originally observed values.

In [67]:
# Remove remaining missing values in analysis variables

analysis_missing_cols = [
    "lat",
    "lon"
]

before = rentals_2025.count()

rentals_2025 = rentals_2025.dropna(
    subset=analysis_missing_cols
)

after = rentals_2025.count()

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)
print("Percentage removed:", round((before - after) / before * 100, 4), "%")

Rows before: 11948
Rows after: 11945
Rows removed: 3
Percentage removed: 0.0251 %


In [71]:
# Create analysis dataset

analysis_cols = [
    "listing_id",
    "suburb",
    "postcode",
    "lat",
    "lon",
    "weekly_rent",
    "bond",
    "available_date",
    "date_listed",
    "days_listed",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "scraped_date",
    "photo_count",
    "video_count",
    "floorplans_count",
    "virtual_tour",
    "property_type",
    "primary_type",
    "secondary_type",
    "available_date_clean",
    "available_weekday",
    "bond_imputed",
    "carspaces_imputed"
]

analysis_data = rentals_2025.select(*analysis_cols)


# Final check for missing value
print("Rows:", analysis_data.count())
print("Columns:", len(analysis_data.columns))

analysis_data.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in analysis_data.columns
]).show(truncate=False)

# Save processed datasets

analysis_data.write.mode("overwrite").parquet(
    "data/analysis_data"
)

Rows: 11945
Columns: 25


+----------+------+--------+---+---+-----------+----+--------------+-----------+-----------+--------+---------+---------+------------+-----------+-----------+----------------+------------+-------------+------------+--------------+--------------------+-----------------+------------+-----------------+
|listing_id|suburb|postcode|lat|lon|weekly_rent|bond|available_date|date_listed|days_listed|bedrooms|bathrooms|carspaces|scraped_date|photo_count|video_count|floorplans_count|virtual_tour|property_type|primary_type|secondary_type|available_date_clean|available_weekday|bond_imputed|carspaces_imputed|
+----------+------+--------+---+---+-----------+----+--------------+-----------+-----------+--------+---------+---------+------------+-----------+-----------+----------------+------------+-------------+------------+--------------+--------------------+-----------------+------------+-----------------+
|0         |0     |0       |0  |0  |0          |0   |0             |0          |0          |0    

In [ ]:
# Create metadata dataset

metadata_cols = [
    "listing_id",
    "address",
    "url",
    "domain_page_id",
    "agent_names",
    "agency",
    "agency_id",
    "structured_features"
]

metadata = rentals_2025.select(*metadata_cols)

# Final check
print("Rows:", metadata.count())
print("Columns:", len(metadata.columns))

# Save metadata
metadata.write.mode("overwrite").parquet(
    "data/metadata"
)

Rows: 11945
Columns: 8
+----------+-------+---+--------------+-----------+------+---------+-------------------+
|listing_id|address|url|domain_page_id|agent_names|agency|agency_id|structured_features|
+----------+-------+---+--------------+-----------+------+---------+-------------------+
|0         |63     |7  |0             |160        |1     |1        |1174               |
+----------+-------+---+--------------+-----------+------+---------+-------------------+

